In [1]:
%pip install tensorflow numpy gensim "scipy<1.13"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:

# EXPERIMENT 5: CBOW MODEL

# 1. Import libraries
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, Lambda, Input
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical
from gensim.models import KeyedVectors

# 2. Input paragraph
data = [
    "natural language processing helps computers understand human language",
    "machine learning helps computers learn patterns from data",
    "deep learning uses neural networks to learn complex patterns",
    "natural language processing uses machine learning techniques",
    "word embeddings represent words as dense numerical vectors",
    "computers understand language through context and word relationships",
    "the cbow model predicts a target word from surrounding context words",
    "word2vec learns word embeddings using context words",
    "neural networks learn useful representations from training data",
    "language models use words and context to understand meaning"
]

# 3. Tokenization
tokenizer = Tokenizer()
tokenizer.fit_on_texts(data)

sequences = tokenizer.texts_to_sequences(data)
vocab_size = len(tokenizer.word_index) + 1

print("Vocabulary Size:", vocab_size - 1)

# 4. Generate context-target pairs
def cbow_model(data, window_size):
    for sentence in data:
        for i in range(window_size, len(sentence) - window_size):
            context = sentence[i-window_size:i] + sentence[i+1:i+window_size+1]
            target = sentence[i]
            yield context, target

window_size = 2

pairs = list(cbow_model(sequences, window_size))

X = np.array([p[0] for p in pairs])
y = to_categorical([p[1] for p in pairs], num_classes=vocab_size)

# 5. Create CBOW model
model = Sequential([
    Input(shape=(window_size * 2,)),
    Embedding(vocab_size, 100),
    Lambda(lambda x: tf.reduce_mean(x, axis=1)),
    Dense(vocab_size, activation='softmax')
])

# 6. Compile and train
model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.fit(X, y, epochs=100, verbose=1)

# 7. Save word vectors
weights = model.get_weights()[0]

with open("vectors.txt", "w") as f:
    f.write(f"{vocab_size - 1} 100\n")

    for word, index in tokenizer.word_index.items():
        vector = ' '.join(map(str, weights[index]))
        f.write(f"{word} {vector}\n")

print("Word vectors saved successfully!")

# 8. Load vectors using Gensim
cbow_output = KeyedVectors.load_word2vec_format(
    "vectors.txt", binary=False
)

# 9. Find similar words
print("Similar words:", cbow_output.most_similar("learning", topn=5))


Vocabulary Size: 48
Epoch 1/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.0000e+00 - loss: 3.8914 
Epoch 2/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.0698 - loss: 3.8793
Epoch 3/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.2326 - loss: 3.8688
Epoch 4/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.4419 - loss: 3.8589
Epoch 5/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.5349 - loss: 3.8491
Epoch 6/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.6512 - loss: 3.8394
Epoch 7/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.6977 - loss: 3.8296
Epoch 8/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.6977 - loss: 3.8196
Epoch 9/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.6977 - loss: 3.8098
Epoch 10/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.7209 - loss: 3.7997
Epoch 11/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.6977 - loss: 3.7897
Epoch 12/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/ste